# GIT Setup


In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ADL/Project/Product-Search

Mounted at /content/drive
/content/drive/MyDrive/ADL/Project/Product-Search


In [3]:
#RUN AT START

!git config --global user.name "emdil99"
!git config --global user.email "emdilauro99@gmail.com"

from google.colab import userdata
token = userdata.get('github_key')
assert token is not None, "GitHub token not found"

repo_url = f"https://{token}@github.com/emdil99/Product-Search.git"
!git remote set-url origin $repo_url
!git pull origin main

print("Git remote updated securely using hidden token.")

From https://github.com/emdil99/Product-Search
 * branch            main       -> FETCH_HEAD
Already up to date.
Git remote updated securely using hidden token.


In [15]:
#RUN TO END CODE SESSION

!git add search_engine.ipynb

#UPDATE COMMIT COMMENT
!git commit -m "Demo Data"


!git push origin main
print("Updated notebook on GitHub")

[main 12c1024] Demo Data
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite search_engine.ipynb (97%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 16.41 KiB | 1.64 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/emdil99/Product-Search.git
   29fb180..12c1024  main -> main
Updated notebook on GitHub


In [4]:
!pip -q install sentence-transformers faiss-cpu
#!pip install -q pyarrow pandas
#!git clone https://github.com/amazon-science/esci-data.git
#!ls esci-data/shopping_queries_dataset

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer as ST
import faiss
import time
from tqdm import tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 100.9 MB/s eta 0:00:00


# Data Structuring

In [5]:
#path = "esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet"
#df = pd.read_parquet(path, filters = [("product_locale", "==", "us")])
#df.columns
#df.head()
#df.to_parquet("eng_products.parquet", index=False)

eng_products = pd.read_parquet("eng_products.parquet")


In [ ]:
eng_products.head()



,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [6]:
#from this data, I want to build the embeddings for the search function based off product title to start
#play around with bullet description in a later variation

products = eng_products[["product_id","product_title"]].drop_duplicates().copy()
products = products.rename(columns={"product_title":"product_text"})
title_by_id = dict(zip(products["product_id"], products["product_text"]))
products.head()

,product_id,product_text
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...


# Embeddings

In [ ]:
model = ST("sentence-transformers/all-mpnet-base-v2")

#sample_texts = products["product_text"].astype(str).head(5).tolist()
#sample_vecs = model.encode(sample_texts, normalize_embeddings=True)

#print("sample_vecs shape:", sample_vecs.shape)
#print("first vector, first 5 numbers:", sample_vecs[0][:5])

In [ ]:
texts = products["product_text"].astype(str).tolist()
batch_size = 256
all_vectors = []


for i in tqdm(range(0,len(texts), batch_size)):
  batch_texts = texts[i:i+batch_size]
  batch_vecs = model.encode(batch_texts, normalize_embeddings=True)
  all_vectors.append(batch_vecs)

product_emb = np.vstack(all_vectors).astype("float32")

np.save("product_emb.npy", product_emb)
products[["product_id"]].to_csv("product_id_map.csv", index=False)
print("done")

100%|██████████| 4750/4750 [13:46<00:00,  5.74it/s]


done


In [ ]:
!ls -lh product_emb.npy product_id_map.csv


-rw------- 1 root root 3.5G Jan  8 15:50 product_emb.npy
-rw------- 1 root root  13M Jan  8 15:50 product_id_map.csv


In [8]:

emb_path = "/content/drive/MyDrive/ADL/Project/Product-Search/product_emb.npy"
id_path  = "/content/drive/MyDrive/ADL/Project/Product-Search/product_id_map.csv"

product_emb = np.load(emb_path)
product_ids = pd.read_csv(id_path)

print("Embeddings:", product_emb.shape, product_emb.dtype)
print("IDs:", product_ids.shape)
print("Rows match?", product_emb.shape[0] == product_ids.shape[0])


Embeddings: (1215854, 768) float32
IDs: (1215854, 1)
Rows match? True


# FAISS

In [9]:

def embed_query(query:str):
  return model.encode([query], normalize_embeddings = True).astype("float32")



def index_search(index, query:str, k: int = 5):
  query_vector = embed_query(query)
  t = time.time()
  D, I = index.search(query_vector, k)
  ms = (time.time() - t) * 1000

  results = []

  for rank, (idx,score) in enumerate(zip(I[0].tolist(), D[0].tolist()), start=1):
    top_prod = product_ids.iloc[idx]["product_id"]
    results.append({
            "rank": rank,
            "product_id": top_prod,
            "title": title_by_id.get(top_prod, ""),
            "score": float(score),
        })
  return results, ms




In [10]:
def build_exact_index(vectors):
    """
        Exact inner product
    """

    embedding_dim = vectors.shape[1]
    index_exact = faiss.IndexFlatIP(embedding_dim)
    index_exact.add(vectors)

    return index_exact


In [ ]:
#TEST

index_exact = build_exact_index(product_emb)
query = "wireless noise cancelling earbuds"

results, ms = index_search(index_exact, query, k=5)

print("Latency (ms):", round(ms, 2))
for r in results:
    print(r["rank"], "|", r["title"], "|", round(r["score"], 3))


Latency (ms): 413.5
1 | Audio-Technica ATH-ANC900BT QuietPoint Wireless Active Noise-Cancelling Headphones | 0.81
2 | Industry Leading Noise Canceling Truly Wireless Earbuds Headset/Headphones with Mic for iOS and Android Phones (Black) | 0.8
3 | Skullcandy Indy ANC True Wireless Noise Cancelling In-Ear Earbud - True Black | 0.785
4 | Bluetooth Sports Earbuds Wireless Earbuds Bluetooth 5.0 True Wireless Bluetooth Earbuds with Charging Case Noise Cancelling | 0.768
5 | Bose QuietComfort 20i Acoustic Noise Cancelling Headphones | 0.763


## HNSW

In [11]:
def build_hnsw_index(vectors, M=32, ef_construction = 200):
    """
        HNSW inner product
        M is graph connectivity, max number of neighbors at each node
        ef_construction is how thoroughly to search for good neighbors
    """

    embedding_dim = vectors.shape[1]
    index_hnsw = faiss.IndexHNSWFlat(embedding_dim,M)
    index_hnsw.metric_type = faiss.METRIC_INNER_PRODUCT
    index_hnsw.hnsw.efConstruction = ef_construction
    index_hnsw.add(vectors)

    return index_hnsw




In [ ]:
#TEST

index_hnsw = build_hnsw_index(product_emb)
index_hnsw.hnsw.efSearch = 64  # query-time knob
query = "wireless noise cancelling earbuds"

results_hnsw, ms_hnsw = index_search(index_hnsw, query, k=5)

print("Latency (ms):", round(ms_hnsw, 2))
for r in results_hnsw:
    print(r["rank"], "|", r["title"], "|", round(r["score"], 3))

Latency (ms): 0.44
1 | Audio-Technica ATH-ANC900BT QuietPoint Wireless Active Noise-Cancelling Headphones | -0.38
2 | Industry Leading Noise Canceling Truly Wireless Earbuds Headset/Headphones with Mic for iOS and Android Phones (Black) | -0.4
3 | Skullcandy Indy ANC True Wireless Noise Cancelling In-Ear Earbud - True Black | -0.431
4 | Bluetooth Sports Earbuds Wireless Earbuds Bluetooth 5.0 True Wireless Bluetooth Earbuds with Charging Case Noise Cancelling | -0.464
5 | Bose QuietComfort 20i Acoustic Noise Cancelling Headphones | -0.473


# DEMO SUBSET FOR STREAMLIT APP


In [12]:
products.columns.tolist()


['product_id', 'product_text']

In [14]:
np.random.seed(0)
subset_n = 200_000

n = product_emb.shape[0]
subset_idx = np.random.choice(n, size=subset_n, replace=False)

product_emb_demo = product_emb[subset_idx]
product_ids_demo = product_ids.iloc[subset_idx].reset_index(drop=True)



demo_pids = set(product_ids_demo["product_id"].tolist())
products_demo = products[products["product_id"].isin(demo_pids)][["product_id", "product_text"]].copy()

DEMO_DIR = "/content/drive/MyDrive/ADL/Project/Product-Search"
import os
os.makedirs(DEMO_DIR, exist_ok=True)

np.save(f"{DEMO_DIR}/product_emb_demo_200k.npy", product_emb_demo.astype("float32"))
product_ids_demo.to_csv(f"{DEMO_DIR}/product_id_map_demo_200k.csv", index=False)
products_demo.to_parquet(f"{DEMO_DIR}/products_demo_200k.parquet", index=False)

print("Saved demo files to:", DEMO_DIR)
!ls -lh $DEMO_DIR



Saved demo files to: /content/drive/MyDrive/ADL/Project/Product-Search
total 4.8G
-rw------- 1 root root 727M Jan  7 17:21 eng_products.parquet
drwx------ 2 root root 4.0K Jan  6 19:53 esci-data
-rw------- 1 root root 586M Jan  9 17:29 product_emb_demo_200k.npy
-rw------- 1 root root 3.5G Jan  8 15:50 product_emb.npy
-rw------- 1 root root  13M Jan  8 15:50 product_id_map.csv
-rw------- 1 root root 2.1M Jan  9 17:29 product_id_map_demo_200k.csv
-rw------- 1 root root  16M Jan  9 17:29 products_demo_200k.parquet
-rw------- 1 root root  290 Dec  5 19:43 README.md
-rw------- 1 root root 3.6K Dec  5 19:44 repo_setup.ipynb
-rw------- 1 root root 131K Jan  9 17:23 search_engine.ipynb
